In [ ]:
import os

import matplotlib.pyplot as plt
import scipy.sparse as sp
import scanpy as sc
import harmonypy as hm
import lamindb as ln

ln.track("3cD0sJzUCR96")

In [ ]:
SWEEP_DIR   = '/home/workspace/PCD_data/xenium/commercial/data/resolution_sweep'
FIGURES_DIR = '/home/workspace/PCD_data/xenium/commercial/figures/processing'

for directory in (SWEEP_DIR, FIGURES_DIR):
    os.makedirs(directory, exist_ok=True)

In [ ]:
def save_fig(fig, stem, description, ulabels=(), features=None, subdir=''):
    output_dir = os.path.join(FIGURES_DIR, subdir)
    os.makedirs(output_dir, exist_ok=True)
    for ax in fig.axes:
        for coll in ax.collections:
            coll.set_rasterized(True)
    for ext in ['png', 'pdf']:
        path = os.path.join(output_dir, f'{stem}.{ext}')
        key = os.path.join(subdir, f'{stem}.{ext}')
        fig.savefig(path, dpi=150, bbox_inches='tight')
        artifact = ln.Artifact(path, key=key, description=description).save()
        if ulabels:
            artifact.ulabels.add(*ulabels)
        if features:
            artifact.features.add_values(features)
    plt.close(fig)

In [ ]:
# ln.Artifact.filter(key__contains='h5ad').df()
# ln.Artifact.filter(key="test_combined_filtered.h5ad").df()

In [ ]:
# Loads the QC-filtered combined object registered by 02_qc_filtering; input tracked automatically.
combined = ln.Artifact.get("KKTKpP4e10Pjv8C70000").load()

# Overwrites .X: rescales every cell to the dataset's median total counts (default target_sum).
sc.pp.normalize_total(combined, inplace=True)
# Changes .X's storage format only (dense to sparse), not its values; most of the 377 genes
# have zero counts in any given cell.
combined.X = sp.csr_matrix(combined.X)
# log1p-transforms .X in place (values become log(count+1)); .X comes from normalize_total above.
sc.pp.log1p(combined)
# Adds layers['lognorm']: snapshot of .X after log1p, before scaling (log-normalized, unscaled).
combined.layers["lognorm"] = combined.X.copy()
# Clips .X values to max 10 and scales each gene to unit variance; zero_center=False keeps .X sparse.
sc.pp.scale(combined, zero_center=False, max_value=10)
# Adds obsm['X_pca']: 30 principal components computed from all 377 genes.
sc.pp.pca(combined, n_comps=30, use_highly_variable=False, random_state=0)

In [ ]:
# Runs Harmony to remove sample-level batch effects; reads obsm['X_pca'], output not yet on combined.
harmony = hm.run_harmony(combined.obsm['X_pca'], combined.obs, vars_use='sample', random_state=0)

In [ ]:
# harmonypy versions differ on whether Z_corr is cells x PCs or PCs x cells.
harmony_pca = harmony.Z_corr
if harmony_pca.shape[0] != combined.n_obs:
    harmony_pca = harmony_pca.T
# Adds obsm['X_pca_harmony']: the batch-corrected coordinates. obsm['X_pca'] stays uncorrected.
combined.obsm['X_pca_harmony'] = harmony_pca

In [ ]:
# Adds obsp['connectivities'] and obsp['distances'] (each cell's 15 nearest neighbors, measured
# in the Harmony space) and uns['neighbors'] (the settings used).
sc.pp.neighbors(combined, use_rep='X_pca_harmony', n_neighbors=15, n_pcs=30, random_state=0)
# Adds obsm['X_umap']: a 2D position per cell computed from the neighbor graph, plus uns['umap'].
sc.tl.umap(combined, random_state=0, spread=2)

In [ ]:
processing_label  = ln.ULabel(name='processing').save()
umap_label        = ln.ULabel(name='umap').save()
clustered_label   = ln.ULabel(name='clustered').save()

In [ ]:
# ln.Artifact.df()

In [ ]:
samples = combined.obs['sample'].unique().tolist()
nrows = 3
ncols = 3
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * nrows, 6 * ncols))
axes = axes.flatten()

for i, metric in enumerate(['total_counts', 'n_genes_by_counts']):
    sc.pl.umap(combined, color=metric, ax=axes[i], size=6, show=False)

sc.pl.umap(combined, color='sample', ax=axes[2], size=6,
           show=False, legend_loc='lower left')

for i, s in enumerate(samples):
    sc.pl.umap(combined, color='sample', groups=[s], ax=axes[i+3],
               size=6, show=False, title=s, na_color='#d3d3d3', legend_loc=None)
    
for ax in axes[3+len(samples):]:
    ax.set_visible(False)

fig.tight_layout()
plt.show()

save_fig(fig,
         stem='umap_qc_sample_overview',
         description='UMAP overview of QC metrics and sample composition, with per-sample highlights.',
         ulabels=(processing_label, umap_label),
         features={'harmony': True, 'spread': 2.0})

In [ ]:
# Register Param schemas once (not inside the loop); these are referenced by features.add_values below.
ln.Param(name='resolution',     dtype=float).save()
ln.Param(name='deg_method',     dtype=str).save()
ln.Param(name='clustering_key', dtype=str).save()
ln.Param(name='n_clusters',     dtype=int).save()
ln.Param(name='harmony',        dtype=bool).save()
ln.Param(name='n_neighbors',    dtype=int).save()
ln.Param(name='n_pcs',          dtype=int).save()
ln.Param(name='spread',         dtype=float).save()

In [ ]:
RESOLUTIONS = [round(0.1 * i, 2) for i in range(1, 12)]  # 0.1, 0.2, …, 1.1
sweep_keys = [f'leiden_res{res:.2f}' for res in RESOLUTIONS]
for res, key in zip(RESOLUTIONS, sweep_keys):
    # Adds an obs column (leiden_res{res}) assigning each cell to a cluster at this resolution.
    sc.tl.leiden(combined, flavor='igraph', n_iterations=2, directed=False,
                 resolution=res, key_added=key, random_state=0)

    # Adds uns['rank_{key}']: marker-gene statistics for each cluster, then writes them to a CSV.
    sc.tl.rank_genes_groups(combined, key, method='wilcoxon', key_added=f'rank_{key}')
    deg = sc.get.rank_genes_groups_df(combined, group=None, key=f'rank_{key}')
    deg_path = os.path.join(SWEEP_DIR, f'deg_{key}_wilcoxon.csv')
    deg.to_csv(deg_path, index=False)  # writes DEG CSV for this resolution to SWEEP_DIR
    artifact = ln.Artifact(deg_path, key=f'deg_{key}_wilcoxon.csv',
                               description=f'DEG markers, leiden res {res:.2f}, Wilcoxon').save()
    artifact.features.add_values({
        'resolution':     res,
        'deg_method':     'wilcoxon',
        'clustering_key': key,
        'n_clusters':     combined.obs[key].nunique()
    })

In [ ]:
fig = sc.pl.umap(combined, color=sweep_keys, wspace=0.05, ncols=3,
                 legend_loc='on data', show=False, return_fig=True, size=6)

plt.show()

save_fig(fig,
         stem='umap_resolution_sweep',
         description='UMAP faceted by all leiden resolutions; used to select final clustering resolution',
         ulabels=(processing_label, umap_label),
         features={'harmony': True, 'spread': 2.0})

In [ ]:
highlight_ncols = 4

for res, key in zip(RESOLUTIONS, sweep_keys):
    clusters = combined.obs[key].cat.categories.tolist()
    n_panels = len(clusters) + 1
    highlight_nrows = (n_panels + highlight_ncols - 1) // highlight_ncols
    fig, axes = plt.subplots(highlight_nrows, highlight_ncols,
                             figsize=(5 * highlight_ncols, 5 * highlight_nrows))
    axes = axes.flatten()

    # Plot only: all clusters labeled on the data. Adds uns['{key}_colors'] if missing (already there from the sweep UMAP).
    sc.pl.umap(combined, color=key, ax=axes[0], size=6, show=False,
               legend_loc='on data', title=f'res {res:.2f}: all clusters')

    for i, cluster in enumerate(clusters):
        # Plot only: one cluster in its color, all other cells gray; reuses uns['{key}_colors'].
        sc.pl.umap(combined, color=key, groups=[cluster], ax=axes[i + 1], size=6, show=False,
                   title=f'res {res:.2f}: cluster {cluster}', na_color='#d3d3d3', legend_loc=None)

    for ax in axes[n_panels:]:
        ax.set_visible(False)

    fig.tight_layout()
    plt.show()

    # Writes cluster_highlights/umap_cluster_highlights_res{res}.png/.pdf under FIGURES_DIR and registers both as lamindb artifacts.
    save_fig(fig,
             stem=f'umap_cluster_highlights_res{res:.2f}',
             description=f'UMAP of leiden res {res:.2f} clusters: all labeled, plus each cluster highlighted individually',
             ulabels=(processing_label, umap_label),
             features={'harmony': True, 'spread': 2.0, 'resolution': res, 'clustering_key': key,
                       'n_clusters': len(clusters)},
             subdir='cluster_highlights')

In [ ]:
artifact = ln.Artifact.from_anndata(combined, key='combined_clustered.h5ad',
                                    description='Combined Xenium object after batch correction, clustering, and resolution sweep').save()
artifact.ulabels.add(processing_label, clustered_label)
artifact.features.add_values({'harmony': True, 'n_neighbors': 15, 'n_pcs': 30, 'spread': 2.0})

ln.finish()

In [ ]:
# ln.Artifact.df()